# Neural ODEs for String Dynamics

**Learning Dynamics with Neural Ordinary Differential Equations**

We start with the nonlinear string equation:
$$\rho A \ddot{u} = (T_0 \Delta - D \Delta \Delta) u - (\sigma_0 + \sigma_1 \Delta) \dot{u} + F_{\text{nonlinear}}(u)$$

The Neural ODE replaces $F_{\text{nonlinear}}(u)$ with an MLP:
- **Time integration** using the Störmer-Verlet scheme
- **Learnable nonlinearity** with an MLP
- **Learnable physical parameters** like string length, tension and damping

In [ ]:
import os
from pathlib import Path

import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax
import soundfile as sf
from IPython.display import Audio, display
from jaxdiffmodal.excitations import create_pluck_modal
from jaxdiffmodal.ftm import (
    evaluate_string_eigenfunctions,
    string_eigenvalues,
    StringParameters,
)
from jaxdiffmodal.time_integrators import (
    solve_sv_leapfrog,
    string_tau_with_density,
)
from jaxtyping import Array, ArrayLike, Float
from tqdm import tqdm
from utils import init_linear_weight

## Utilities

In [ ]:
def create_static_filter(
    model,
    static_params_lambda,
):
    is_static_filter = jax.tree_util.tree_map(lambda _: False, model)

    selected_params = static_params_lambda(model)

    if isinstance(selected_params, tuple):
        true_values = tuple(True for _ in selected_params)
    else:
        # Single parameter case
        true_values = True

    is_static_filter = eqx.tree_at(
        static_params_lambda,
        is_static_filter,
        true_values,
    )
    return is_static_filter


def visualize_results(
    model,
    time: Array,
    n_steps_vis: int,
    n_steps_train: int,
    dt: float,
    n_modes: int,
    losses: Array | None = None,
):
    """Visualize training results and model predictions."""

    time_test = jnp.arange(n_steps_vis) * dt

    # Get modal trajectories for visualization
    targ_test_traj_modal: Array = gt_model(
        n_steps=n_steps_vis,
        dt=dt,
        u0=u0,
        v0=v0,
        return_modal=True,
    )

    pred_test_traj_modal = model(
        n_steps=n_steps_vis,
        dt=dt,
        u0=u0,
        v0=v0,
        return_modal=True,
    )

    pred_test_traj_phys: Array = model(
        n_steps=n_steps_vis,
        dt=dt,
        u0=u0,
        v0=v0,
        return_modal=False,
    )

    # Target physical position
    targ_test_traj_phys: Array = targ_test_traj_modal[..., 0] @ gt_model.weights

    # Check if losses exist and training is complete
    plot_losses = losses is not None and len(losses) > 0

    # Create plots - adjust subplot layout based on whether we're plotting losses
    if plot_losses:
        fig = plt.figure(figsize=(15, 10))
        gs_main = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)
        # Plot loss in bottom right when training is complete
        loss_ax = fig.add_subplot(gs_main[1, 1])
        loss_ax.semilogy(losses)
        loss_ax.set_title("Training Loss")
        loss_ax.set_xlabel("Epoch")
        loss_ax.set_ylabel("MSE Loss")
        loss_ax.grid(True)
    else:
        fig = plt.figure(figsize=(15, 5))
        gs_main = fig.add_gridspec(1, 2, hspace=0.3, wspace=0.3)

    # Plot 1: Physical space comparison
    physical_ax = fig.add_subplot(gs_main[0, 0])
    physical_ax.plot(
        time_test[:n_steps_vis],
        targ_test_traj_phys[:n_steps_vis],
        "b-",
        label="Target",
    )
    physical_ax.plot(
        time_test[:n_steps_vis],
        pred_test_traj_phys[:n_steps_vis],
        "r--",
        label="Neural ODE",
    )
    physical_ax.set_title("Physical Space Displacement")
    physical_ax.set_xlabel("Time (s)")
    physical_ax.set_ylabel("Displacement (m)")
    physical_ax.set_ylim(-0.01, 0.01)
    physical_ax.grid(True)
    physical_ax.axvline(
        x=n_steps_train * dt,
        color="k",
        alpha=1.0,
        label="Training Horizon",
    )
    physical_ax.legend(loc="upper right")

    # Plot 2: Modal amplitudes comparison - use the right half of top row
    if plot_losses:
        # Create subplot within the top-right area, avoiding the loss plot
        modal_gs = gs_main[0, 1].subgridspec(3, 1, hspace=0.4)
    else:
        # Use the right half for modal plots
        modal_gs = gs_main[0, 1].subgridspec(3, 1, hspace=0.4)

    for mode_idx in range(min(3, n_modes)):
        modal_ax = fig.add_subplot(modal_gs[mode_idx, 0])
        modal_ax.plot(
            time_test,
            targ_test_traj_modal[:n_steps_vis, mode_idx, 0],
            label="Target",
            alpha=0.8,
            linewidth=1.5,
        )
        modal_ax.plot(
            time_test,
            pred_test_traj_modal[:n_steps_vis, mode_idx, 0],
            "--",
            label="Prediction",
            alpha=0.8,
            linewidth=1.5,
        )
        modal_ax.set_title(f"Mode {mode_idx + 1}", fontsize=10)
        if mode_idx == 2:  # Only bottom plot gets x-label
            modal_ax.set_xlabel("Time (s)", fontsize=9)
        modal_ax.set_ylabel("Amplitude", fontsize=9)
        modal_ax.set_ylim(-0.01, 0.01)
        modal_ax.tick_params(labelsize=8)
        if mode_idx == 0:  # Only top plot gets legend
            modal_ax.legend(fontsize=8)
        modal_ax.grid(True, alpha=0.3)
        modal_ax.axvline(
            x=n_steps_train * dt,
            color="k",
            alpha=0.7,
            linestyle=":",
        )

    plt.show()

## Generate Synthetic String Data

First, we'll create synthetic string data using `jaxdiffmodal`
for the nonlinear string.


In [ ]:

n_modes: int = 15
sample_rate: int = 22050
dt: float = 1.0 / sample_rate
n_steps_train: int = 441
n_steps_test: int = 882
n_steps_train_vis: int = 1000
n_steps_total_vis = 2000

epochs = 5_000
learning_rate = 5e-3

In [ ]:

string_params = StringParameters()
indices = jnp.arange(n_modes) + 1

lambda_mu = string_eigenvalues(
    n_modes,
    string_params.length,
)
exc = create_pluck_modal(
    lambdas=lambda_mu,
    string_length=string_params.length,
    initial_deflection=0.03,
)

weights = evaluate_string_eigenfunctions(
    indices=indices,
    position=jnp.array(0.1),
    params=string_params,
)

u0 = jnp.array(exc)
v0 = jnp.zeros_like(u0)
time = jnp.arange(n_steps_train) * dt


class StringModel(eqx.Module):
    length: ArrayLike
    d3_with_density: ArrayLike
    log_Ts0_with_density: ArrayLike
    bending_stiffness_with_density: ArrayLike
    tau_with_density: ArrayLike
    weights: Array  # Modal weights for single position output
    mlp: eqx.Module | None = None
    n_modes: int = 10

    def __call__(
        self,
        n_steps: int,
        dt: float,
        u0: Array,
        v0: Array,
        return_modal: bool = False,
    ) -> Float[Array, " n_steps"] | Float[Array, "n_steps n_modes n_states"]:
        # Unpack parameters
        length: ArrayLike = self.length
        d3_with_density: ArrayLike = self.d3_with_density
        # Convert from log-space
        Ts0_with_density: ArrayLike = jnp.exp(self.log_Ts0_with_density)
        bending_stiffness_with_density: ArrayLike = self.bending_stiffness_with_density
        tau_with_density: ArrayLike = self.tau_with_density

        # get the analytical eigenvalues
        lambda_mu: Array = string_eigenvalues(
            self.n_modes,
            length,
        )

        # get the damping and stiffness terms
        omega_mu_squared: Array = (
            bending_stiffness_with_density * lambda_mu**2 + Ts0_with_density * lambda_mu
        )
        gamma2_mu: Array = d3_with_density * lambda_mu

        # calculate the factor for the nonlinear term
        string_norm: float = string_params.length / 2
        string_tau: Array = tau_with_density * lambda_mu / string_norm

        # print("string_tau:", string_tau.shape)
        def nl_fn(q: ArrayLike) -> Array:
            return lambda_mu * q * (string_tau @ q**2)

        def nl_fn_nn(q: ArrayLike) -> Array:
            return lambda_mu * self.mlp(q)

        _, out_pos, out_vel = solve_sv_leapfrog(
            gamma2_mu=gamma2_mu,
            omega_mu_squared=omega_mu_squared,
            u0=u0,
            v0=v0,
            dt=dt,
            n_steps=n_steps,
            nl_fn=nl_fn_nn if self.mlp is not None else nl_fn,
        )

        if return_modal:
            return jnp.stack([out_pos, out_vel], axis=-1)
        else:
            # Apply weights to get single position output
            return out_pos @ self.weights

In [ ]:

u0 = jnp.array(exc)
v0 = jnp.zeros_like(u0)
string_tau: float = string_tau_with_density(string_params)

gt_model = StringModel(
    length=string_params.length,
    log_Ts0_with_density=jnp.log(string_params.Ts0 / string_params.density),
    d3_with_density=(string_params.d3 / string_params.density),
    bending_stiffness_with_density=(
        string_params.bending_stiffness / string_params.density
    ),
    tau_with_density=string_tau_with_density(string_params),
    weights=weights,
    mlp=None,
    n_modes=n_modes,
)


# Get weighted trajectory at single position for training target
targ_test_traj: Array = gt_model(
    n_steps=n_steps_train,
    dt=dt,
    u0=u0,
    v0=v0,
    return_modal=True,
)

# slice a section for training
targ_train_traj: Array = targ_test_traj[:n_steps_train]

In [ ]:

# Initialize model with random weights for optimization
key = jax.random.PRNGKey(3407)
(
    key_len,
    key_Ts0,
    key_d3,
    key_mlp,
) = jax.random.split(key, 4)

model = StringModel(
    # length=string_params.length,
    # log_Ts0_with_density=jnp.log(string_params.Ts0 / string_params.density),
    # d3_with_density=(string_params.d3 / string_params.density),
    bending_stiffness_with_density=(
        string_params.bending_stiffness / string_params.density
    ),
    tau_with_density=string_tau_with_density(string_params),
    length=jax.random.uniform(
        shape=(1,),
        minval=0.6,
        maxval=0.8,
        key=key_len,
    ),
    log_Ts0_with_density=jax.random.uniform(
        shape=(1,),
        minval=jnp.log(10_000),
        maxval=jnp.log(80_000),
        key=key_Ts0,
    ),
    d3_with_density=jax.random.uniform(
        shape=(1,),
        minval=5.0,
        maxval=7.0,
        key=key_d3,
    ),
    # mlp=init_linear_weight(
    #     eqx.nn.MLP(
    #         in_size=n_modes,
    #         out_size=n_modes,
    #         width_size=128,
    #         depth=4,
    #         activation=jax.nn.selu,
    #         key=key_mlp,
    #     ),
    #     jax.nn.initializers.lecun_normal(),
    #     key_mlp,
    # ),
    weights=weights,
    n_modes=n_modes,
)

# Create the static filter using the wrapper function
# Comment out parameters you want to train
is_static_filter = create_static_filter(
    model=model,
    static_params_lambda=lambda m: (
        # m.length,
        # m.log_Ts0_with_density,
        # m.d3_with_density,
        m.tau_with_density,
        m.weights,
        m.bending_stiffness_with_density,
    ),
)

# Now, partition the model using our custom filter
static_model, diff_model = eqx.partition(
    model,
    is_static_filter,
)


def generate_and_save_audio(
    model,
    gt_model,
    u0: Array,
    v0: Array,
    dt: float,
    sample_rate: int,
    n_steps: int = 44100,
    output_dir: str = "tmp_node",
    file_prefix: str = "",
):
    pred_traj = model(
        n_steps=n_steps,
        dt=dt,
        return_modal=True,
        u0=u0,
        v0=v0,
    )
    targ_traj: Array = gt_model(
        n_steps=n_steps,
        dt=dt,
        u0=u0,
        v0=v0,
        return_modal=True,
    )

    targ_velocity = targ_traj[..., 1] @ gt_model.weights
    pred_velocity = pred_traj[..., 1] @ model.weights

    # Display audio
    display(Audio(targ_velocity, rate=sample_rate))
    display(Audio(pred_velocity, rate=sample_rate))

    # Save normalized audio files
    os.makedirs(output_dir, exist_ok=True)

    target_file = f"{output_dir}/{file_prefix}target_audio.wav"
    pred_file = f"{output_dir}/{file_prefix}predicted_audio.wav"

    sf.write(
        target_file,
        targ_velocity / jnp.abs(targ_velocity).max(),
        sample_rate,
    )
    sf.write(
        pred_file,
        pred_velocity / jnp.abs(pred_velocity).max(),
        sample_rate,
    )

    return targ_velocity, pred_velocity


# Generate initial audio comparison
generate_and_save_audio(
    model=model,
    gt_model=gt_model,
    u0=u0,
    v0=v0,
    dt=dt,
    sample_rate=sample_rate,
    file_prefix="initial_",
)

Define the training loop and loss function.

In [ ]:
def save_animation_frame(
    model,
    time: Array,
    weights: Array,
    frame_idx: int,
    gt_model,
    output_dir: str = "tmp_node",
    n_steps_vis: int = 882,
):
    Path(output_dir).mkdir(exist_ok=True, parents=True)

    time_test = jnp.arange(n_steps_vis) * dt

    targ_test_traj_modal: Array = gt_model(
        n_steps=n_steps_vis,
        dt=dt,
        u0=u0,
        v0=v0,
        return_modal=True,
    )

    pred_test_traj_modal = model(
        n_steps=n_steps_vis,
        dt=dt,
        u0=u0,
        v0=v0,
        return_modal=True,
    )

    pred_test_traj_phys: Array = pred_test_traj_modal[..., 0] @ gt_model.weights
    targ_test_traj_phys: Array = targ_test_traj_modal[..., 0] @ gt_model.weights

    # Create figure with centered plot and table underneath
    fig = plt.figure(figsize=(12, 6))
    gs = fig.add_gridspec(2, 1, height_ratios=[4, 1], hspace=0.4)

    # Physical space comparison
    physical_ax = fig.add_subplot(gs[0, 0])
    physical_ax.plot(
        time_test,
        targ_test_traj_phys[:n_steps_vis],
        "b-",
        label="Target",
    )
    physical_ax.plot(
        time_test,
        pred_test_traj_phys[:n_steps_vis],
        "r--",
        label="Optim",
    )
    physical_ax.set_title("Physical Space Displacement")
    physical_ax.set_xlabel("Time (s)")
    physical_ax.set_ylabel("Displacement (m)")
    physical_ax.set_ylim(-0.005, 0.005)
    physical_ax.legend(loc="upper right")
    physical_ax.grid(True)

    # Add parameter table
    table_ax = fig.add_subplot(gs[1, 0])
    table_ax.axis("off")

    # Create table data - handle both JAX arrays and floats
    def format_param(param):
        return param.item() if hasattr(param, "item") else param

    table_data = [
        ["Parameter", "Current", "Ground Truth"],
        [
            "Length",
            f"{format_param(model.length):.4f}",
            f"{format_param(gt_model.length):.4f}",
        ],
        [
            r"$\hat{d}_3$",
            f"{format_param(model.d3_with_density):.6f}",
            f"{format_param(gt_model.d3_with_density):.6f}",
        ],
        [
            r"$\hat{T}_0$",
            # Show actual value, not log
            f"{format_param(jnp.exp(model.log_Ts0_with_density)):.1f}",
            f"{format_param(jnp.exp(gt_model.log_Ts0_with_density)):.1f}",
        ],
    ]

    table = table_ax.table(
        cellText=table_data,
        cellLoc="center",
        loc="center",
        colWidths=[0.25, 0.25, 0.25],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 2)

    # Style the header row
    for i in range(len(table_data[0])):
        table[(0, i)].set_facecolor("#40466e")
        table[(0, i)].set_text_props(weight="bold", color="white")

    plt.tight_layout()
    plt.savefig(f"{output_dir}/frame_{frame_idx:05d}.png", dpi=150, bbox_inches="tight")
    plt.close()

In [ ]:
def train_neural_ode(
    model,
    epochs,
    save_frames=False,
    frame_interval: int = 100,
    window_size: int = 10,
    n_windows: int = 4096,
):
    @eqx.filter_jit
    def training_step(
        model,
        optimizer,
        opt_state,
        targ_train_traj,
        key,
        window_size,
        n_windows,
    ):
        @eqx.filter_value_and_grad
        def loss_fn(
            diff_model,
            static_model,
            targ_train_traj,
            key,
        ):
            model: StringModel = eqx.combine(diff_model, static_model)

            total_steps = targ_train_traj.shape[0]
            max_start = total_steps - window_size

            random_starts = jax.random.randint(
                key,
                (n_windows,),
                0,
                max_start + 1,
            )

            def compute_window_loss(start_idx):
                window = jax.lax.dynamic_slice_in_dim(
                    targ_train_traj,
                    start_idx,
                    1,
                    axis=0,
                ).squeeze(axis=0)

                # Predict trajectory from these initial conditions
                pred_window: Array = model(
                    n_steps=window_size,
                    dt=dt,
                    u0=window[:, 0],
                    v0=window[:, 1],
                    return_modal=True,
                )

                # Get corresponding ground truth window using dynamic slice
                targ_window = jax.lax.dynamic_slice_in_dim(
                    targ_train_traj,
                    start_idx,
                    window_size,
                    axis=0,
                )
                # window has shape (n_slices, n_steps, n_modes, 2)
                # pred_window_physical = jnp.einsum(
                #     "nmv,m->nv", pred_window, model.weights
                # )
                # targ_window_physical = jnp.einsum(
                #     "nmv,m->nv", targ_window, model.weights
                # )

                # Compute MSE loss for this window
                window_loss = jnp.mean((pred_window - targ_window) ** 2)
                # window_loss = jnp.mean(
                #     (pred_window_physical - targ_window_physical) ** 2
                # )
                return window_loss

            # Use vmap to compute losses for all random windows in parallel
            window_losses = jax.vmap(compute_window_loss)(random_starts)
            total_loss = jnp.mean(window_losses)

            return total_loss

        static_model, diff_model = eqx.partition(
            model,
            is_static_filter,
        )
        loss_value, grads = loss_fn(
            diff_model,
            static_model,
            targ_train_traj,
            key,
        )

        updates, opt_state = optimizer.update(grads, opt_state)
        model = eqx.apply_updates(model, updates)
        return model, opt_state, loss_value

    schedule = optax.cosine_onecycle_schedule(
        transition_steps=epochs,
        peak_value=learning_rate,
    )
    optimizer = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adam(schedule),
    )
    opt_state = optimizer.init(
        eqx.filter(model, eqx.is_array),
    )

    losses = []

    train_key = jax.random.PRNGKey(42)

    bar = tqdm(range(epochs))
    for epoch in bar:
        train_key, epoch_key = jax.random.split(train_key)

        window_size = min(window_size, targ_train_traj.shape[0] - 1)

        model, opt_state, loss_value = training_step(
            model,
            optimizer,
            opt_state,
            targ_train_traj,
            epoch_key,
            window_size,
            n_windows,
        )
        losses.append(loss_value)

        # Early stopping if NaN detected or loss explodes
        if jnp.isnan(loss_value) or loss_value > 1e8:
            print(
                f"\nWarning: Training stopped early at epoch {epoch + 1} "
                "due to instability"
            )
            print(f"Loss value: {loss_value}")
            break

        bar.set_description(f"Epoch {epoch + 1}/{epochs} | Loss: {loss_value:.6f}")

        if save_frames and epoch % frame_interval == 0:
            save_animation_frame(
                model=model,
                time=time,
                weights=weights,
                frame_idx=epoch // frame_interval,
                gt_model=gt_model,
            )

    return model, losses

In [ ]:
# First visualisation of the initial model
visualize_results(
    model=model,
    time=time,
    n_steps_vis=n_steps_total_vis,
    n_steps_train=n_steps_train_vis,
    dt=dt,
    n_modes=n_modes,
)

In [ ]:

trained_model, training_losses = train_neural_ode(
    model,
    epochs=epochs,
    save_frames=True,
    frame_interval=100,
)
print(f"Training completed! Final loss: {training_losses[-1]:.6f}")

In [ ]:
visualize_results(
    model=trained_model,
    time=time,
    n_steps_vis=n_steps_total_vis,
    n_steps_train=n_steps_train_vis,
    dt=dt,
    n_modes=n_modes,
)

# Generate and save trained model audio
generate_and_save_audio(
    model=trained_model,
    gt_model=gt_model,
    u0=u0,
    v0=v0,
    dt=dt,
    sample_rate=sample_rate,
    file_prefix="trained_",
)

In [ ]:
print(gt_model.length, trained_model.length)
print(gt_model.log_Ts0_with_density, trained_model.log_Ts0_with_density)
print(gt_model.d3_with_density, trained_model.d3_with_density)


def play_intensity_demo(model: StringModel, b3_tension: float = 60.97):
    """Demo showcasing B3 with increasing initial condition intensities"""

    intensities = [0.1, 0.3, 0.5, 0.7, 0.9, 1.1, 1.2, 1.3]
    start_times = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]
    note_durations = [1, 1, 1, 1, 1, 1, 1, 1]

    note_tracks = []

    for i, intensity in enumerate(intensities):
        start_time = start_times[i]
        note_duration = note_durations[i]
        n_steps_note = int(note_duration * sample_rate)

        model_tmp = eqx.tree_at(
            lambda p: p.log_Ts0_with_density,
            model,
            jnp.log(b3_tension / string_params.density),
        )

        pred_traj = model_tmp(
            n_steps=n_steps_note,
            dt=dt,
            u0=u0 * intensity,
            v0=v0,
            return_modal=True,
        )

        note_audio = pred_traj[..., 1] @ model_tmp.weights
        note_tracks.append((start_time, note_audio))

    max_end_time = max(
        start_times[i] + note_durations[i] for i in range(len(intensities))
    )
    total_samples = int(max_end_time * sample_rate)
    full_audio = jnp.zeros(total_samples)

    for start_time, note_audio in note_tracks:
        start_sample = int(start_time * sample_rate)
        end_sample = min(start_sample + len(note_audio), total_samples)
        audio_length = end_sample - start_sample

        full_audio = full_audio.at[start_sample:end_sample].add(
            note_audio[:audio_length]
        )

    display(Audio(full_audio, rate=sample_rate))

    return full_audio


# Run the intensity demo
play_intensity_demo(gt_model)
play_intensity_demo(trained_model)